In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import deeplake as dl
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

In [ ]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")

In [ ]:
# Global Variables and Configs
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Image Constants
IMG_HEIGHT = 224
IMG_WIDTH = 407
IMG_CHANNELS = 3
IMAGE_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

GLOBAL_BACKBONE = keras.applications.MobileNetV2(
    weights='imagenet', 
    include_top=False, 
    pooling='avg', 
    input_shape=IMAGE_SHAPE
)
GLOBAL_BACKBONE.trainable = False

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [4]:
# Preprocessing Detection

def image_preprocessing(image):
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    image = data['images']
    labels = data['labels']  # Integer Class IDs from VisDrone (0 to 9)
    
    # 1. Process the image
    image = image_preprocessing(image)
    
    # 2. Track Humans: Pedestrians (1) OR People (2)
    is_human_mask = tf.logical_or(tf.equal(labels, 1), tf.equal(labels, 2))
    human_count = tf.reduce_sum(tf.cast(is_human_mask, tf.float32))
    
    # 3. Track Vehicles: Car (4), Vans (5), Trucks (6), Buses (9), and Motorcycles (10)
    is_car_mask = (tf.equal(labels, 4) | 
                   tf.equal(labels, 5) | 
                   tf.equal(labels, 6) | 
                   tf.equal(labels, 9) | 
                   tf.equal(labels, 10))
    vehicle_count = tf.reduce_sum(tf.cast(is_car_mask, tf.float32))
    
    # 4. Combine into a single multi-count vector: [human_count, car_count]
    counts_vector = tf.stack([human_count, vehicle_count], axis=0)
    
    return image, counts_vector

# Feature Stream Pipelines

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .cache()
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .cache()
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

In [ ]:
# Preprocessing City Labels

extract_train = (train_ds.tensorflow()
                 .map(lambda x: image_preprocessing(x['images']), num_parallel_calls=tf.data.AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

extract_val = (val_ds.tensorflow()
               .map(lambda x: image_preprocessing(x['images']), num_parallel_calls=tf.data.AUTOTUNE)
               .batch(BATCH_SIZE)
               .prefetch(AUTOTUNE))

extract_test = (test_ds.tensorflow()
                .map(lambda x: image_preprocessing(x['images']), num_parallel_calls=tf.data.AUTOTUNE)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

def classification_preprocessing(data_stream, pca_model=None, kmeans_model=None):
    features = GLOBAL_BACKBONE.predict(data_stream, verbose=0)

    if pca_model is None: # Training mode
        pca_model = PCA(n_components=50, random_state=42)
        kmeans_model = MiniBatchKMeans(n_clusters=14, random_state=42, n_init=5, batch_size=8192)
        labels = kmeans_model.fit_predict(pca_model.fit_transform(features))
        return tf.constant(labels, dtype=tf.int32), pca_model, kmeans_model
        
    # Inference mode
    labels = kmeans_model.predict(pca_model.transform(features))
    return tf.constant(labels, dtype=tf.int32)

# Training phase: captures the trained PCA and KMeans weights
tf_train_labels, trained_pca, trained_kmeans = classification_preprocessing(extract_train)

# Validation and Test phases: reuse those exact trained weights
tf_val_labels = classification_preprocessing(extract_val, pca_model=trained_pca, kmeans_model=trained_kmeans)
tf_test_labels = classification_preprocessing(extract_test, pca_model=trained_pca, kmeans_model=trained_kmeans)

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE)

In [6]:
# Adding Pipelines

def shape_structure_outputs(img_batch, count_batch, label_batch):
    # Enforce static ranks for the Keras graph builder
    img_batch.set_shape([None, *IMAGE_SHAPE])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])   # [Batch_Size]
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [7]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(IMAGE_SHAPE), name='input_image')

    base_model = keras.applications.MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.Conv2D(32, (1, 1), activation='relu', kernel_initializer='he_normal')(shared_features)
    x_detect = layers.GlobalAveragePooling2D()(x_detect)
    
    x_detect = layers.Dense(128, activation=None, kernel_initializer='he_normal')(x_detect)
    x_detect = layers.BatchNormalization()(x_detect)
    x_detect = layers.LeakyReLU(negative_slope=0.1)(x_detect)
    
    # Final count output layer
    count_output = layers.Dense(2, activation='softplus', kernel_initializer='he_normal', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    
    protected_counts = layers.Lambda(lambda x: tf.stop_gradient(x))(count_output)

    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, protected_counts])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "mean_absolute_error"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 1.0, 
            "count_output": 0.1
        }
    )
    return model

china_model = build_multi_head_model()

history = china_model.fit(train_pipeline, validation_data=val_pipeline, epochs=7)

C:\Users\etito\AppData\Local\Temp\ipykernel_12928\2456054677.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(



Epoch 1/7
    405/Unknown 300s 729ms/step - city_output_accuracy: 0.5217 - city_output_loss: 1.4434 - count_output_loss: 20.2670 - count_output_mae: 20.2671 - loss: 3.4701

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


405/405 ━━━━━━━━━━━━━━━━━━━━ 327s 795ms/step - city_output_accuracy: 0.6443 - city_output_loss: 1.0114 - count_output_loss: 16.9837 - count_output_mae: 17.0355 - loss: 2.7179 - val_city_output_accuracy: 0.8394 - val_city_output_loss: 0.4560 - val_count_output_loss: 12.7830 - val_count_output_mae: 13.2441 - val_loss: 1.7882
Epoch 2/7
405/405 ━━━━━━━━━━━━━━━━━━━━ 265s 656ms/step - city_output_accuracy: 0.7875 - city_output_loss: 0.5641 - count_output_loss: 12.0628 - count_output_mae: 12.1012 - loss: 1.7754 - val_city_output_accuracy: 0.8741 - val_city_output_loss: 0.3404 - val_count_output_loss: 12.1001 - val_count_output_mae: 12.5420 - val_loss: 1.5994
Epoch 3/7
405/405 ━━━━━━━━━━━━━━━━━━━━ 327s 805ms/step - city_output_accuracy: 0.8257 - city_output_loss: 0.4671 - count_output_loss: 11.0316 - count_output_mae: 11.0673 - loss: 1.5748 - val_city_output_accuracy: 0.8796 - val_city_output_loss: 0.3327 - val_count_output_loss: 11.9239 - val_count_output_mae: 12.3713 - val_loss: 1.5754
Epoch

In [8]:
# Testing Model

results = china_model.evaluate(test_pipeline, return_dict=True)
print(results)

for test_images, test_targets in test_pipeline.take(1):
    
    # Generate predictions for this batch
    city_preds, count_preds = china_model.predict(test_images)
    
    # Extract the highest probability index
    predicted_city_indices = np.argmax(city_preds, axis=-1)
    true_city_indices = test_targets["city_output"].numpy().flatten()
    true_counts = test_targets["count_output"].numpy()
    
    print("\nBatch Sample Breakdown:")
    for i in range(5):
        # Map integer indices back to string labels
        pred_label = city_labels[predicted_city_indices[i]]
        true_label = city_labels[true_city_indices[i]]
        
        print(f"\nSample {i+1}:")
        print(f"  [City] Predicted: {pred_label:<12} | True: {true_label}")
        print(f"  [Count] Predicted: {count_preds[i]} | True: {true_counts[i]}")

101/101 ━━━━━━━━━━━━━━━━━━━━ 66s 649ms/step - city_output_accuracy: 0.8640 - city_output_loss: 0.3230 - count_output_loss: 15.2341 - count_output_mae: 15.3871 - loss: 1.8654
{'city_output_accuracy': 0.8639751672744751, 'city_output_loss': 0.32301124930381775, 'count_output_loss': 15.234084129333496, 'count_output_mae': 15.387099266052246, 'loss': 1.8654060363769531}
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step

Batch Sample Breakdown:

Sample 1:
  [City] Predicted: jinchang     | True: jinchang
  [Count] Predicted: [ 6.982553 20.616613] | True: [ 0. 15.]

Sample 2:
  [City] Predicted: guangzhou    | True: guangzhou
  [Count] Predicted: [10.541661 60.14991 ] | True: [ 0. 29.]

Sample 3:
  [City] Predicted: suzhou       | True: suzhou
  [Count] Predicted: [23.273497 46.410103] | True: [54. 48.]

Sample 4:
  [City] Predicted: hong-kong    | True: hong-kong
  [Count] Predicted: [17.543407 45.27902 ] | True: [12. 33.]

Sample 5:
  [City] Predicted: shaoxing     | True: shaoxing
  [Count] Predicted: 